# Standard Stimulus Creation
Description: This notebook contains to build reliable stimuli that are easy to use, keep track of and that provides sufficient informations for uses in a further future.


In [ ]:
%reload_ext autoreload
%autoreload 2

# standard packages
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# project packages
import params
import utils

## I— How to create a vec file with the right sequence keys

This notebook needs **one thing** from your vec file: its **last column** must give a *sequence key* to every trigger. The other columns are ignored here.

### The key structure

Each key is a whole number read as two parts:

```
<sequence-type digits><repetition digits>
                      └── the last n_digit_for_rep digits (4 by default)
```

- **Sequence type** (leading digits): *what* was shown — e.g. grating direction, image id, condition number.
- **Repetition** (last `n_digit_for_rep` digits): *which presentation* of that sequence type, zero-padded.

Example with `n_digit_for_rep = 4`, sequence type = grating direction (1–8):

| Direction | Repetition | Key |
|-----------|-----------|-----|
| 1 | 0 | `10000` |
| 1 | 1 | `10001` |
| 2 | 0 | `20000` |
| 8 | 3 | `80003` |

### The rules (important!)

1. **One row per trigger.** The very first line of the file is treated as a header and dropped, so it must *not* be a real trigger.
2. **Same key for every trigger of one repetition.** All triggers making up direction 1, repetition 0 carry the key `10000`.
3. **Start sequence types at 1, not 0.** Keys are stored as whole numbers, so a leading zero is lost (`01000` becomes `1000`). The first digit must be non-zero.
4. **Number repetitions from 0**, zero-padded to `n_digit_for_rep` digits. Repetition `0000` of each sequence type **must exist** — it is used as the timing reference.
5. **Keep every key contiguous.** All triggers sharing a key must form a single uninterrupted block in the file. Never reuse the same key for triggers that appear in two separate places — they would be merged into one (much-too-long) repetition. *(This is the most common mistake.)*
6. **Use the same total width for every key** (e.g. always 5 digits) so the split between sequence type and repetition stays consistent.

Repetitions do **not** need to be in order: you can interleave sequence types and repetitions however your stimulus did, as long as each key stays a single contiguous block (rule 5).

### A worked example to copy

`ressources/add_standard_keys_to_dg_vec.ipynb` builds exactly these keys for an 8-direction, 4-repetition drifting-grating stimulus (each repetition is 12 s = 600 triggers at 50 Hz). Use it as a template: it reads a raw vec, writes the sequence key into the last column following the rules above, and saves the result.

## II — Generating a stimulus `.bin` file

A stimulus is described by **two** files that work as a pair:

| file | what it holds |
|------|---------------|
| `.vec` | one row per trigger; column 1 says **which frame** of the `.bin` to show (see Appendix A for the last column) |
| `.bin` | the **frames themselves** — the actual images sent to the DMD |

So the `.vec` is a playlist and the `.bin` is the image library. This appendix is about writing the `.bin`.

### The `BinFile` helper

Reading and writing `.bin` files goes through `BinFile`:

```python
from utils.binfile import BinFile     # note: not re-exported as utils.BinFile
```

- `BinFile.read_header(path)` → `{'xsize', 'ysize', 'nb_images', 'nb_bits'}` without opening the whole file.
- **Reading**: `BinFile(path, 0, 0, rig_id, mode="r")`, then `read_frame(i)` gives a 2-D array of floats in `[0, 1]` (the frame size comes from the header, which is why the two zeros are ignored). `len(obj)` / `obj.nb_frames` give the frame count.
- **Writing**: `BinFile(path, xsize, ysize, rig_id, nb_images=N, mode="w")`, then `append(frame)` for each frame (again floats in `[0, 1]`), then `close()`.

Two things that bite people:

1. **`nb_images` must be the final frame count.** The header is written when the file is opened, so it cannot be corrected afterwards. Appending a different number of frames leaves a file whose header lies about its own length.
2. **`rig_id` matters.** Each rig has its own display polarity and optical path, and frames are stored *pre-corrected* for it. `BinFile` applies that correction automatically from `params.rig_params` — so write with the rig you will actually display on (`params.MEA`). Reading a file back with the same `rig_id` returns the image you started from.

### The `F` convention — always the second frame

By convention the **second frame of the bin (index 1) is a letter `F`**.

The letter F is strongly asymmetric, so *any* unwanted flip or rotation of the display is immediately visible — unlike a circle or a checkerboard, which look fine mirrored. Keeping it at a **fixed, known index** means any `.vec` can run an orientation check simply by pointing its frame-index column at `1`, without knowing anything else about the stimulus.

So when you build a stimulus: frame 0 = **mid-grey** (a `0.5` reference frame, handy to eyeball the stimulus' mean luminance), **frame 1 = the F**, frames 2+ = your actual stimulus.

### Check it before you record

Once the `.bin` and `.vec` exist, preview them with the **StimulusDisplayer** tool (its own repository, kept next to this pipeline). It replays the vec against the bin and shows what the DMD would display, so you can confirm the frame order, the sequence keys, the shutter/colour columns — and the F.

- Point it at your `.bin` and `.vec`, choose your MEA, and you should see an **upright F**. If it appears mirrored or rotated, the display orientation is wrong — fix that *before* recording, because it silently flips every receptive field you compute afterwards.
- `StimulusDisplayer/make_F_test.py` generates a ready-made F-only stimulus (`F_TEST_432x432.bin` + `.vec`) if you just want to test what I would look like on the rig without building a full stimulus.


In [ ]:
from utils.binfile import BinFile

# ---- Inputs ------------------------------------------------------------
# Stimuli live in params.stim_directory (RessourcesAndTools/StandardVec). Change the name.
bin_path = os.path.join(params.stim_directory, "my_stimulus.bin")
frame_size = 432  # square stimulus window, in pixels
rig_id = params.MEA  # frames are stored pre-corrected for THIS rig


def make_F(size):
    """A white upright 'F' on black: asymmetric, so any flip/rotation shows."""
    margin, thick = size // 5, size // 8
    frame = np.zeros((size, size))
    frame[margin : size - margin, margin : margin + thick] = 1.0  # vertical stroke
    frame[margin : margin + thick, margin : size - margin] = 1.0  # top stroke
    mid = size // 2
    frame[mid - thick // 2 : mid + thick // 2, margin : size - margin - thick] = 1.0
    return frame


# ---- Build the frames --------------------------------------------------
frames = [
    np.full(
        (frame_size, frame_size), 0.5
    ),  # frame 0 : mid-grey (mean-luminance reference)
    make_F(frame_size),  # frame 1 : the F test (convention)
    # frames 2+ : add your own stimulus frames here, as arrays of floats in [0, 1]
]

# ---- Write the .bin ----------------------------------------------------
# nb_images must be the FINAL number of frames (the header is written upfront).
binf = BinFile(
    bin_path, frame_size, frame_size, rig_id, nb_images=len(frames), mode="w"
)
for frame in frames:
    binf.append(frame)
binf.close()
print(f"Wrote {len(frames)} frames to {bin_path}")
print(BinFile.read_header(bin_path))

# ---- Read it back and check the F looks right --------------------------
reader = BinFile(bin_path, 0, 0, rig_id, mode="r")
fig, axs = plt.subplots(1, len(reader), figsize=(4 * len(reader), 4))
for i, ax in enumerate(np.atleast_1d(axs)):
    ax.imshow(reader.read_frame(i), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"frame {i}" + ("  <- F test" if i == 1 else ""), fontsize=14)
    ax.axis("off")
reader.close()
plt.show()

## III — The standard preamble: grey, F, and the four squares

Every stimulus should begin with the same short **preamble** so each recording carries its
own display checks.

**Standard bin** — the first six frames are always:

| frame | content |
|-------|---------|
| 0 | grey (mean-luminance reference) |
| 1 | **F** — orientation test (asymmetric, so any flip / rotation is obvious) |
| 2–5 | the four squares: top, right, bottom, left |

Your real stimulus frames then start at frame **6**.

**Standard vec** — always play the **F briefly**, then the **four squares**, *before* the
real stimulus. The squares are flashed one at a time just outside the MEA (top, right,
bottom, left); analysed in `A_Standard_Vec_Analysis.ipynb` they check that the display is
spatially consistent — a cell whose receptive field is at the top responds most to the top
square.

**Reserved sequence ids** — the preamble uses ids in the `999x` block so they never collide
with a real stimulus's ids (which start at 1): F = `9999`, and squares `9991` (top), `9992`
(right), `9993` (bottom), `9994` (left). To prepend the preamble to a real stimulus, keep
the stimulus's own `1..N` ids and **offset its frame indices by 6** (the six standard frames
come first in the bin).

The cell below writes this preamble as its own stimulus (`standard_preamble.bin` +
`_std.vec`) — run it as a standalone check recording, or use it as the block to prepend to
your stimuli.

In [ ]:
# ---- parameters --------------------------------------------------------
rig_id = params.MEA
frame_size = params.get_rig_params(rig_id)["size_dmd"][0]  # square DMD frame, in pixels
refresh_hz = 40  # DMD refresh you display it at (triggers per second)
f_duration_s = 0.5  # how long the F is shown
square_duration_s = 1.0  # how long each square is shown
n_square_reps = 20  # repetitions of the 4-square cycle

# ---- frames: grey, F, and the four squares -----------------------------
frames = utils.standard_bin_frames(
    frame_size, mea=rig_id
)  # [grey, F, top, right, bottom, left]
bin_path = os.path.join(params.stim_directory, "standard_preamble.bin")
binf = BinFile(
    bin_path, frame_size, frame_size, rig_id, nb_images=len(frames), mode="w"
)
for frame in frames:
    binf.append(frame)
binf.close()
print(f"Wrote {len(frames)} frames to {bin_path}")

# ---- vec: play F briefly, then the four squares (reserved ids) ----------
vec_rows = utils.standard_preamble_vec_rows(
    refresh_hz=refresh_hz,
    f_duration_s=f_duration_s,
    square_duration_s=square_duration_s,
    n_square_reps=n_square_reps,
)
header = [
    0,
    len(vec_rows),
    0,
    0,
    0,
]  # first line = summary (col 1 = total frames), dropped on load
vec_path = os.path.join(params.stim_directory, "standard_preamble_std.vec")
np.savetxt(vec_path, np.vstack([header, vec_rows]), fmt="%d")
print(
    f"Wrote {len(vec_rows)} rows to {vec_path}  ({n_square_reps} reps of the 4 squares)"
)

# ---- preview the four squares + the MEA footprint ----------------------
fraction = utils.mea_extent_on_display(rig_id)
squares = utils.make_four_squares_frames(frame_size, fraction)
fig, axs = plt.subplots(1, len(squares), figsize=(4 * len(squares), 4))
for ax, (name, frame) in zip(axs, squares):
    ax.imshow(frame, cmap="gray", vmin=0, vmax=1, extent=[-1, 1, -1, 1])
    ax.add_patch(
        plt.Rectangle(
            (-fraction, -fraction),
            2 * fraction,
            2 * fraction,
            fill=False,
            edgecolor="red",
            lw=1.5,
        )
    )
    ax.set_title(f"{name} square", fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
axs[0].set_ylabel("red box = MEA", fontsize=12)
plt.show()

## IV — Prepend the preamble to your stimulus (the easy way)

By convention every recording should begin with the preamble. Once you have generated your
stimulus's `.bin` and `_std.vec` (from your stimulus-making tool), this one call adds the
standard preamble in front and writes the ready-to-display files:

- your stimulus frames are **copied byte-for-byte** (unchanged) after the six standard frames,
- the vec plays the **F + four squares** (reserved ids) first, then your stimulus, with your
  stimulus's frame indices shifted and its own sequence ids untouched.

Point it at your files, pick the output names, and display the result on the rig.

In [ ]:
# Your existing stimulus (from your stimulus-making tool), for THIS rig (params.MEA):
stimulus_bin = os.path.join(params.stim_directory, "my_stimulus.bin")
stimulus_vec = os.path.join(params.stim_directory, "my_stimulus_std.vec")

# Where to write the preamble-prefixed files to display on the rig:
output_bin = os.path.join(params.stim_directory, "my_stimulus_with_preamble.bin")
output_vec = os.path.join(params.stim_directory, "my_stimulus_with_preamble_std.vec")

utils.prepend_standard_preamble(
    stimulus_bin,
    stimulus_vec,
    output_bin,
    output_vec,
    mea=params.MEA,
    n_square_reps=1,  # keep it short when prepending to every recording (a few reps = a few seconds)
)